# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list all record sets available in the dataset and, for each, show its fields and columns by `@id`. This will help identify what data you can extract and manipulate.

In [ ]:
# List record sets and their fields/columns
record_sets = dataset.record_sets
print(f"Available Record Sets (@id): {[rs.id for rs in record_sets]}")
for rs in record_sets:
    print(f"\nRecord Set: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else ''}")
    print(f"  Data type: {rs.data_type if hasattr(rs, 'data_type') else ''}")
    # List fields by @id and name
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            field_name = getattr(field, 'name', '')
            print(f"    - {field.id} ({field_name})")
    # List columns by @id and name if available
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col in rs.columns:
            col_name = getattr(col, 'name', '')
            print(f"    - {col.id} ({col_name})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We choose the first record set (if available) as an example.

In [ ]:
# Specify record set(s) by @id as found above
record_sets = dataset.record_sets
record_set_ids = [rs.id for rs in record_sets]

# Prepare DataFrames for each record set
dataframes = {}
for rec_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded {len(df)} rows for record set: {rec_id}")
    except Exception as e:
        print(f"Could not load records for {rec_id}: {e}")

# Show columns and head of the first available record set
if record_set_ids:
    first_id = record_set_ids[0]
    if first_id in dataframes:
        print(f"Columns in {first_id}: {dataframes[first_id].columns.tolist()}")
        display(dataframes[first_id].head())
    else:
        print(f"No data for {first_id}.")
else:
    print("No record sets found in dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Assuming the dataset has fields such as protein abundance or molecular weight, we will use those for numeric analysis. Remember to reference columns by their `@id` as shown in the overview above.

In [ ]:
import numpy as np

# Set record_set_id and numeric_field based on the IDs from section 2 above.
# Replace these example names with actual IDs as printed above.

if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Try to guess a numeric field by data type or column name
    possible_numeric_fields = [c for c in df.columns if df[c].dtype.kind in 'fi' or 'abundance' in c.lower() or 'weight' in c.lower() or 'count' in c.lower()]
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
        print(f"Using numeric field {numeric_field} for demonstration.")
    else:
        print("No obvious numeric field found. Adjust 'numeric_field' manually.")
        numeric_field = df.columns[0]

    # Remove rows with missing values in our numeric field
    filtered_df = df[df[numeric_field].notnull()]

    # Try to convert to float for numeric analysis
    filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
    filtered_df = filtered_df[filtered_df[numeric_field].notnull()]

    # Example filter: greater than median
    threshold = filtered_df[numeric_field].median()
    filtered2_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (median value):")
    print(filtered2_df.head())

    # Normalize
    filtered2_df[f"{numeric_field}_normalized"] = (filtered2_df[numeric_field] - filtered2_df[numeric_field].mean()) / filtered2_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered2_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    # Pick the next available column as a group if exists
    possible_group_field = [c for c in df.columns if c != numeric_field]
    if possible_group_field:
        group_field = possible_group_field[0]
        print(f"\nGrouping by field: {group_field}")
        grouped_df = filtered2_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        print(grouped_df.head())
else:
    print("No suitable record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

This example uses matplotlib and seaborn for visualization. Adjust the fields as necessary based on the actual record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field
if record_set_ids and possible_numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If we also have a categorical (group) field, plot mean values
    if possible_group_field:
        top_groups = filtered_df[group_field].value_counts().index[:5]
        subset = filtered_df[filtered_df[group_field].isin(top_groups)]
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=subset)
        plt.title(f"{numeric_field} distribution by {group_field}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to:

1. Load dataset metadata and records directly from a Croissant schema URL.
2. Inspect available record sets and their field/column structure (referenced by `@id`).
3. Load and explore tabular data with pandas DataFrames.
4. Apply basic EDA and common data wrangling operations on numeric columns, including filtering and normalization.
5. Visualize distributions and groupings to spot trends and outliers.

For further exploration, reference additional record sets, fields, and their `@id`s as required. The exact structure of the FAIR^2 dataset may provide further insights with detailed downstream analysis for protein chemistry, biomarker discovery, or experimental validation.